In [116]:
import pandas as pd
from vertexai.generative_models import (
    FunctionDeclaration,
    GenerationConfig,
    GenerativeModel,
    Tool,
    HarmCategory,
    HarmBlockThreshold
)
import json
import re


In [117]:
import import_ipynb
from filters import get_all_filters

get Boolean filters, relevant stages and relevant materials

In [118]:
boolean_filter_json, stage_filter, materials  = get_all_filters()
product_filter_json = json.loads(boolean_filter_json)


Get a sample of ConstructConnect Data

In [119]:
construct_connect_data = pd.read_pickle('data/construct_connect_data.pkl')


Filter by stage

In [120]:
stages_to_include = stage_filter["high"] + stage_filter["moderate"]
filtered_by_stage = construct_connect_data[construct_connect_data['Stage'].isin(stages_to_include)]


Filter by Value

In [121]:
if filtered_by_stage['Valuation_Value'].dtype == 'object':  # Check if it's a string (object type)
    filtered_by_stage.loc[:, 'Valuation_Value'] = pd.to_numeric(filtered_by_stage.loc[:, 'Valuation_Value'], errors='coerce')
filtered_cc_data = filtered_by_stage[filtered_by_stage['Valuation_Value'] > 1000000]


In [122]:
filtered_cc_data[:200].to_csv('data/filtered_cc_data.csv', index=False)

In [123]:
cc_data_subset = filtered_cc_data[:3]


Gemini Flash 2.0

In [124]:
PROJECT_ID = "proj-sales-recommender-dev"
LOCATION = "us-central1" 

import vertexai

vertexai.init(project=PROJECT_ID, location=LOCATION)


In [125]:
MODEL_ID = "gemini-2.0-flash-001"

model = GenerativeModel(
    MODEL_ID,
    safety_settings={
            HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_NONE,
            HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_NONE,
            HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_NONE,
            HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: HarmBlockThreshold.BLOCK_NONE,
        },

)

Single query

In [126]:
def run_prompt(product, search_query, cc_project_json, materials, product_priority):
    question = f'''

    **Objective:** Classify ConstructConnect projects as very high, high, moderate, low, very low and not relevant.

    

    **Instructions:**

    1. **Analyze the provided JSON data** representing ConstructConnect projects to understand relevant fields and data.
    2. **Utilize the Search Query** corresponding to the product (provided below) to identify if project is related to the given product.
    3. **Utilize the relevant materials** to identify if the project has any of the materials listed.
    4. **Utilize the product priority** to weigh the classifying attributes accordingly.
    4. **Classify projects:**
        * With a project description that includes the product category or related terms.
        * Where the product query matches across mutliple project fields.
        * Containing sunstantial number of matching materials from the **Relevant Materials** list.
        * Based on the Product Priority and project-product relationship.
        * If all else is equal, assign higher priority to higher value projects.
    4. **Estimate the relevancy of each project** based on product matches, materials and total dollar amount. 
    5. **Respond with a list of project evaluations** in valid JSON as provided in the Example Output below. Classify the projects as very high, high, moderate, low, very low and not relevant. RETURN ONLY THE JSON RESPONSE.


    **JSON Project Data:** 
    {cc_project_json}

    **Product Category:**
    {product}

    **Boolean Filter for the Product:**
    {search_query}

    **Relevant Materials:**
    {materials}

    ** Product Priority:**
    {product_priority}

    **Example Output:**
    {{"ProjectID": 1006193703,"Product": {product}, "Product priority":{product_priority}, "Materials matching": [material1, material2, material3],  "Relevance Classification": High, "Reasoning": reason for classification}}

    '''

    prompt = question
    contents = [prompt]

    # Generate text using non-streaming method
    response = model.generate_content(contents)

    # Print generated text and usage metadata
    #print(f"\nAnswer:\n{response.text}")
    return response.text

In [127]:
product_responses = []
for index, row in cc_data_subset.iterrows():  # Iterate through rows
    row_json_str = row.to_json()  # Convert row to dictionary
    cc_project_json_record = json.loads(row_json_str) 
    for item in product_filter_json:
        output = run_prompt(item['Filter'], item['Query'], cc_project_json_record, materials, item['Priority'] )
        otpt = output.replace("```json", "").replace("```", "")
        product_responses.append(otpt)



In [128]:
product_responses

['\n[\n  {\n    "ProjectID": 1006872527,\n    "Product": "3M Sales",\n    "Product priority": "low",\n    "Materials matching": [\n      "Acoustical Ceilings",\n      "Acoustical Walls",\n      "Cold Formed Metal Framing",\n      "Drywall/Gypsum",\n      "Firestopping",\n      "Insulation",\n      "Lath & Plaster",\n      "Metal Decking",\n      "Metal Doors",\n      "Waterproofing"\n    ],\n    "Relevance Classification": "low",\n    "Reasoning": "Project is a police department addition/renovation. While it includes several relevant materials such as Acoustical Ceilings, Drywall, Metal Doors and Firestopping, the product \'3M Sales\' doesn\'t have a strong direct association. The project includes \'Addition / Renovation\' which indicates relevance but product priority is low. No explicit mention of 3M products."\n  }\n]\n',
 '\n[\n  {\n    "ProjectID": 1006872527,\n    "Product": "Access Door Sales",\n    "Product priority": "low",\n    "Materials matching": [\n      "Acoustical Ceili

In [129]:
all_data = []
for item in product_responses:
    try:
        # Load each string as JSON, handling potential errors
        json_list = json.loads(item)
        all_data.extend(json_list)  # Extend the list with the parsed dictionaries
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON: {e}")
        print(f"Problematic string: {item}")
        continue # Skip to the next string if there's an error


df_single_prompt_output = pd.DataFrame(all_data)


In [130]:
df_single_prompt_output.to_csv('data/single_prompt_output.csv', index=False)

Multiple prompts

In [131]:
def run_prompt_boolean_filter(product, search_query, cc_project_json, product_priority):
    question = f''' 
    **Objective:** Identify if the ConstructConnect project is related to the given product.

    **Instructions:**
        1. **Analyze the provided JSON data** representing ConstructConnect project to understand relevant fields and data.
        2. **Utilize the Boolean filters** corresponding to the **Product Category** to identify if the project is related to the given product by matching across multpile project fields.
        3. **Respond as YES or NO with a reason for your answer in a valid JSON as provided in the Example Output below. RETURN ONLY THE JSON RESPONSE.**

    **JSON Project Data:** 
    {cc_project_json}

    **Product Category:**
    {product}
   
    **Boolean Filters:**
    {search_query}

    **Example Output:**
    [
    {{"ProjectID": 1006193703, "Product": {product}, "Product priority":{product_priority}, "Project related to Product": YES, "Product Reasoning": reason for relation response}},
    ]

    '''

    prompt = question
    contents = [prompt]

    # Generate text using non-streaming method
    response = model.generate_content(contents)
    return response.text


In [132]:
def run_prompt_material_filter(materials, cc_project_json):
    question = f''' 
    **Objective:** Identify if the ConstructConnect project has any of the given relevant materials.

    **Instructions:**
        1. **Analyze the provided JSON data** representing ConstructConnect project to understand relevant fields and data.
        2. **Utilize the Materials list** to identify if the project has any of the materials listed by matching across multpile project fields.
        3. **Respond as YES or NO with a list of all the materials mentioned in your answer in a valid JSON as provided in the Example Output below. RETURN ONLY THE JSON RESPONSE.**

    **JSON Project Data:** 
    {cc_project_json}

    **Materials List:**
    {materials}
   

    **Example Output:**
    {{"ProjectID": 1006193703, "Relevant materials present": YES, "Materials": ["Material1","Material2", ...], "Reasoning": reason for relation response}},
    '''

    prompt = question
    contents = [prompt]

    # Generate text using non-streaming method
    response = model.generate_content(contents)
    return response.text

In [133]:
def run_prompt_relevance (cc_project_json, materials):
    question = f''' 
    **Objective:** Classify ConstructConnect projects as very high, high, moderate, low, very low and not relevant.

    **Instructions:**
        1. **Analyze the provided JSON data** representing ConstructConnect project, the product it is related to, materials used and the valuation.
        2. **Utilize the Relevant Materials list** provided. THis list contains materials that are very important for the company.
        3. **Utilize the Product Priority** to weight the classifying attributes accordingly.
        4. **Estimate the relevance of the project using the specifed product** based on the materials and the total dollar amount. 
        5. **Respond with a list of project evaluations** in valid JSON as provided in the Example Output below. Classify the projects as very high, high, moderate, low, very low and not relevant. RETURN ONLY THE JSON RESPONSE.

    **JSON Project Data:** 
    {cc_project_json}

    **Materials List:**
    {materials}
   
    **Product Priority:**
    {cc_project_json['Product priority']}

    **Example Output:**
    {{"ProjectID": 1006193703, "Relevance classification": , "Reasoning": reason for classification}}
    '''

    prompt = question
    contents = [prompt]

    # Generate text using non-streaming method
    response = model.generate_content(contents)
    return response.text

In [134]:
def run_separate_prompts(cc_projects_json_records, product_filter_json, materials):
    material_response = run_prompt_material_filter(materials, cc_projects_json_records)
    material_response_json = json.loads(material_response.replace("```json", "").replace("```", ""))
   
    product_responses = []
    for item in product_filter_json:
        output = run_prompt_boolean_filter(item['Filter'], item['Query'], cc_projects_json_records, item['Priority'])
        product_responses.append(output.replace("```json", "").replace("```", "").replace("[", "").replace("]", ""))
    return product_responses, material_response_json, 

In [135]:
def create_dataframe(json_strings, material_data):
    """Creates a Pandas DataFrame from JSON strings and adds material data."""
    data = []
    for json_str in json_strings:
        try:
            cleaned_json_str = json_str.strip()
            if cleaned_json_str:
              json_data = json.loads(cleaned_json_str)
              data.append(json_data)
        except json.JSONDecodeError as e:
            print(f"Error decoding JSON: {e}")
            print(f"Problematic string: {json_str}")
            return None
        except Exception as e:
            print(f"An error occurred: {e}")
            return None

    if not data:
      return None

    df = pd.DataFrame(data)

    # Add the material columns.  This is the key change:
    df["Relevant materials present"] = material_data["Relevant materials present"]
    df["Materials"] = [material_data["Materials"]] * len(df) # Important: Make it a list of lists
    df["Materials Reasoning"] = [material_data["Reasoning"]] * len(df) # Important: Make it a list of lists


    return df

In [136]:
parameters_df = pd.DataFrame()
for index, row in cc_data_subset.iterrows():  # Iterate through rows
    row_json_str = row.to_json()  # Convert row to dictionary
    cc_project_json_record = json.loads(row_json_str) 
    print(cc_project_json_record)

    product_responses, material_response = run_separate_prompts(cc_project_json_record, product_filter_json, materials)
    df = create_dataframe(product_responses, material_response)
    df["Valuation"] = [cc_project_json_record["Valuation_Value"]]*len(df)
    parameters_df = pd.concat([parameters_df, df], ignore_index=True)

{'ProjectID': 1006872527, 'DataSourceID': 'US', 'Title': 'Agawam Police Department Addition/Renovation', 'Stage': 'Low Bids Announced', 'URL': 'http://insight.cmdgroup.com/SingleSignOn/ProjectDetails/1006872527/1/', 'UpdateDate': '2023-08-28', 'IsProspective': False, 'UpdateText': 'Project reviewed, Stage confirmed as Low Bids Announced', 'Valuation_Value': 4637163.0, 'Valuation_Currency': 'USD', 'Valuation_ValueType': 'Confirmed Value', 'Parameters_Parameter_Ownership': 'City', 'Parameters_Parameter_WorkType': 'Addition/Alteration', 'Parameters_Parameter_CommenceDate': '2023-10-20', 'Parameters_Parameter_FloorArea': 18000.0, 'Parameters_Parameter_Structures': 2.0, 'Parameters_Parameter_Units': None, 'Parameters_Parameter_BidDate': '2023-08-22', 'Parameters_Parameter_FloorAreaUnitofMeasure': 'Square Feet', 'Parameters_Parameter_IsSingleTrade': False, 'DocumentAvailability_Plans': True, 'DocumentAvailability_Specs': True, 'DocumentAvailability_Addenda': True, 'ParentCategories_PrimaryCa

In [137]:
parameters_df

,ProjectID,Product,Product priority,Project related to Product,Product Reasoning,Relevant materials present,Materials,Materials Reasoning,Valuation
0,1006872527,3M Sales,low,NO,The provided data does not contain any direct ...,YES,"[Waterproofing, Roofing, Acoustical Ceilings, ...",The project details mention the following mate...,4637163.0
1,1006872527,Access Door Sales,low,NO,The provided project data does not contain any...,YES,"[Waterproofing, Roofing, Acoustical Ceilings, ...",The project details mention the following mate...,4637163.0
2,1006872527,Ceiling Sales,high,YES,"The Details_Detail_Scope mentions ""CEILINGS:Ro...",YES,"[Waterproofing, Roofing, Acoustical Ceilings, ...",The project details mention the following mate...,4637163.0
3,1006872527,Certainteed Drywall Sales,low,YES,The project details mention 'Drywall/Gypsum' u...,YES,"[Waterproofing, Roofing, Acoustical Ceilings, ...",The project details mention the following mate...,4637163.0
4,1006872527,Cultured Stone Sales,low,YES,The project details mention 'Stone' and 'Stone...,YES,"[Waterproofing, Roofing, Acoustical Ceilings, ...",The project details mention the following mate...,4637163.0
5,1006872527,Drywall Sales,high,YES,The project details mention 'Drywall/Gypsum' u...,YES,"[Waterproofing, Roofing, Acoustical Ceilings, ...",The project details mention the following mate...,4637163.0
6,1006872527,EIFS & Stucco Sales,high,YES,The project scope mentions 'Exterior Insulatio...,YES,"[Waterproofing, Roofing, Acoustical Ceilings, ...",The project details mention the following mate...,4637163.0
7,1006872527,FRP Sales,high,YES,The Details_Detail_Scope field mentions 'FLOOR...,YES,"[Waterproofing, Roofing, Acoustical Ceilings, ...",The project details mention the following mate...,4637163.0
8,1006872527,Fry Reglet Sales,high,NO,The provided project data does not contain any...,YES,"[Waterproofing, Roofing, Acoustical Ceilings, ...",The project details mention the following mate...,4637163.0
9,1006872527,Georgia Pacific Drywall Sales,low,YES,The project details mention 'Drywall/Gypsum' i...,YES,"[Waterproofing, Roofing, Acoustical Ceilings, ...",The project details mention the following mate...,4637163.0


In [138]:
parameters_df.to_csv('data/first_pass_output.csv', index=False)

In [139]:
import pandas as pd
import json

def classify_relevance(df):
  
    df["Relevance Classification"] = ""  # Initialize the new column

    for index, row in df.iterrows():
        if row["Project related to Product"] == "NO" or row["Relevant materials present"] == "NO":
            df.loc[index, "Relevance Classification"] = "not relevant"
            df.loc[index, "Classification Reasoning"] = 'Project is not related to the product or relevant materials are not present'
        else:
            product_data = {
                "ProjectID": row["ProjectID"],
                "Product": row["Product"],
                "Product priority": row["Product priority"],
                "Materials": row["Materials"],
                "Valuation": row["Valuation"]
            }
            #json_data = json.loads(product_data)

            relevance_classification = run_prompt_relevance(product_data, materials)
            relevance_classification_json = json.loads(relevance_classification.replace("```json", "").replace("```", "").replace("[", "").replace("]", ""))
            df.loc[index, "Relevance Classification"] = relevance_classification_json['Relevance classification']
            df.loc[index, "Classification Reasoning"] = relevance_classification_json['Reasoning']

    return df


In [140]:
classifications_df = classify_relevance(parameters_df)
classifications_df

,ProjectID,Product,Product priority,Project related to Product,Product Reasoning,Relevant materials present,Materials,Materials Reasoning,Valuation,Relevance Classification,Classification Reasoning
0,1006872527,3M Sales,low,NO,The provided data does not contain any direct ...,YES,"[Waterproofing, Roofing, Acoustical Ceilings, ...",The project details mention the following mate...,4637163.0,not relevant,Project is not related to the product or relev...
1,1006872527,Access Door Sales,low,NO,The provided project data does not contain any...,YES,"[Waterproofing, Roofing, Acoustical Ceilings, ...",The project details mention the following mate...,4637163.0,not relevant,Project is not related to the product or relev...
2,1006872527,Ceiling Sales,high,YES,"The Details_Detail_Scope mentions ""CEILINGS:Ro...",YES,"[Waterproofing, Roofing, Acoustical Ceilings, ...",The project details mention the following mate...,4637163.0,very high,The project includes multiple high-priority ma...
3,1006872527,Certainteed Drywall Sales,low,YES,The project details mention 'Drywall/Gypsum' u...,YES,"[Waterproofing, Roofing, Acoustical Ceilings, ...",The project details mention the following mate...,4637163.0,moderate,The project includes multiple relevant materia...
4,1006872527,Cultured Stone Sales,low,YES,The project details mention 'Stone' and 'Stone...,YES,"[Waterproofing, Roofing, Acoustical Ceilings, ...",The project details mention the following mate...,4637163.0,moderate,Project includes multiple relevant materials s...
5,1006872527,Drywall Sales,high,YES,The project details mention 'Drywall/Gypsum' u...,YES,"[Waterproofing, Roofing, Acoustical Ceilings, ...",The project details mention the following mate...,4637163.0,very high,"The project includes Drywall/Gypsum, Lath & Pl..."
6,1006872527,EIFS & Stucco Sales,high,YES,The project scope mentions 'Exterior Insulatio...,YES,"[Waterproofing, Roofing, Acoustical Ceilings, ...",The project details mention the following mate...,4637163.0,high,The project involves several key materials rel...
7,1006872527,FRP Sales,high,YES,The Details_Detail_Scope field mentions 'FLOOR...,YES,"[Waterproofing, Roofing, Acoustical Ceilings, ...",The project details mention the following mate...,4637163.0,very high,The project includes several high-priority mat...
8,1006872527,Fry Reglet Sales,high,NO,The provided project data does not contain any...,YES,"[Waterproofing, Roofing, Acoustical Ceilings, ...",The project details mention the following mate...,4637163.0,not relevant,Project is not related to the product or relev...
9,1006872527,Georgia Pacific Drywall Sales,low,YES,The project details mention 'Drywall/Gypsum' i...,YES,"[Waterproofing, Roofing, Acoustical Ceilings, ...",The project details mention the following mate...,4637163.0,moderate,Project includes relevant materials like Acous...


In [141]:
classifications_df.to_csv('data/second_pass_output.csv', index=False)